### Loading the dataset

In [1]:
with open("../dickens/combined.txt", "r", encoding='utf-8') as f:
    text = f.read()

print(text[:1000])      
print(f"length of dataset in chars: {len(text)}")

The Project Gutenberg eBook, Three Ghost Stories, by Charles Dickens


This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re-use it under the terms of the Project Gutenberg License included
with this eBook or online at www.gutenberg.org





Title: Three Ghost Stories


Author: Charles Dickens



Release Date: March 9, 2013  [eBook #1289]
[This file was first posted on April 5, 1998]

Language: English

Character set encoding: UTF-8


***START OF THE PROJECT GUTENBERG EBOOK THREE GHOST STORIES***


Transcribed from the 1894 Chapman and Hall edition of “Christmas Stories”
by David Price, email ccx074@pglaf.org





                           THREE GHOST STORIES


                            by Charles Dickens




CONTENTS

The Haunted House             121
The Trial For Murder          303
The Signal-Man                312




THE HAUNTED HOUSE.
IN TWO CHAPTERS. {121}


                                 [


### Building character tokens    

In [2]:
characters = sorted(list(set(text)))
print(f"characters: {''.join(characters)}")
vocab_size = len(characters)
print(f"vocab size: {vocab_size}")

# first time using those functions
# enumerate returns a list?/object containing pairs of number/value
stoi = {ch:i for i, ch in enumerate(characters)}
# we're creating two dicts dynamically, in a:b a is the key and b is the value
itos = {i:ch for i, ch in enumerate(characters)}

encode = lambda s: [stoi[c] for c in s] # encode a string, looping through its char elts
decode = lambda s: ''.join(itos[c] for c in s) # take a list of ints, output a string

print(f"encoding of hello world: f{encode('hello world')}")
print(f"decoding of [69, 66, 73, 73, 76, 1, 84, 76, 79, 73, 65]: {decode([69, 66, 73, 73, 76, 1, 84, 76, 79, 73, 65])}")
print(f"sanity check: {decode(encode(''.join(characters)))}")

characters: 
 !"#$%&'()*,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_abcdefghijklmnopqrstuvwxyz{}~ £½Ôàáâæçèéêëíîòóôöāěŏœ—‘’“”﻿
vocab size: 120
encoding of hello world: f[69, 66, 73, 73, 76, 1, 84, 76, 79, 73, 65]
decoding of [69, 66, 73, 73, 76, 1, 84, 76, 79, 73, 65]: hello world
sanity check: 
 !"#$%&'()*,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_abcdefghijklmnopqrstuvwxyz{}~ £½Ôàáâæçèéêëíîòóôöāěŏœ—‘’“”﻿


### Storing into a tensor

In [3]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([24350792]) torch.int64
tensor([119,  52,  69,  66,   1,  48,  79,  76,  71,  66,  64,  81,   1,  39,
         82,  81,  66,  75,  63,  66,  79,  68,   1,  66,  34,  76,  76,  72,
         12,   1,  52,  69,  79,  66,  66,   1,  39,  69,  76,  80,  81,   1,
         51,  81,  76,  79,  70,  66,  80,  12,   1,  63,  86,   1,  35,  69,
         62,  79,  73,  66,  80,   1,  36,  70,  64,  72,  66,  75,  80,   0,
          0,   0,  52,  69,  70,  80,   1,  66,  34,  76,  76,  72,   1,  70,
         80,   1,  67,  76,  79,   1,  81,  69,  66,   1,  82,  80,  66,   1,
         76,  67,   1,  62,  75,  86,  76,  75,  66,   1,  62,  75,  86,  84,
         69,  66,  79,  66,   1,  62,  81,   1,  75,  76,   1,  64,  76,  80,
         81,   1,  62,  75,  65,   1,  84,  70,  81,  69,   0,  62,  73,  74,
         76,  80,  81,   1,  75,  76,   1,  79,  66,  80,  81,  79,  70,  64,
         81,  70,  76,  75,  80,   1,  84,  69,  62,  81,  80,  76,  66,  83,
         66,  79,  14,   1,  

### Train/val separation

In [4]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

### Block setup

In [5]:
block_size = 16
train_data[:block_size+1]

# all possible examples:
# x as input, y as target
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    print(f"when context: {x[:t+1]}, target: {y[t]}")


when context: tensor([119]), target: 52
when context: tensor([119,  52]), target: 69
when context: tensor([119,  52,  69]), target: 66
when context: tensor([119,  52,  69,  66]), target: 1
when context: tensor([119,  52,  69,  66,   1]), target: 48
when context: tensor([119,  52,  69,  66,   1,  48]), target: 79
when context: tensor([119,  52,  69,  66,   1,  48,  79]), target: 76
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76]), target: 71
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76,  71]), target: 66
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76,  71,  66]), target: 64
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76,  71,  66,  64]), target: 81
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76,  71,  66,  64,  81]), target: 1
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76,  71,  66,  64,  81,   1]), target: 39
when context: tensor([119,  52,  69,  66,   1,  48,  79,  76,  71,  66,  64,  81,   1,  39])

### Dataloader

In [6]:
block_size = 16 #context length
batch_size = 4 #independent sequences to process in parallel

def get_batch(split):
    data = train_data if split == 'train' else val_data
    # here, params are high (upper limit for sampling) and size, that is the number of elts to sample
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    # target range, which is why we're not just sampling 1 at a time
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y
    
xb, yb = get_batch("train")
print("inputs:")
print(xb.shape)
print(xb)
print("targets:")
print(yb.shape)
print(yb)

for b in range (batch_size):
    for t in range (block_size):
        print(f"when context is {xb[b, :t+1]} target is {yb[b, t]}")

inputs:
torch.Size([4, 16])
tensor([[ 72,  66,  79, 116,  80,   1,  78,  82,  66,  80,  81,  70,  76,  75,
           1,  69],
        [ 75,  65,   1,  84,  66,  75,  81,   1,  62,  84,  62,  86,   1,  81,
          69,  79],
        [ 62,  81,   1,  76,  75,  66,   1,  62,  75,  76,  81,  69,  66,  79,
           1,  82],
        [ 65,  70,  68,  70,  76,  82,  80,   1,  64,  62,  80,  81,  73,  66,
          12,   1]])
targets:
torch.Size([4, 16])
tensor([[ 66,  79, 116,  80,   1,  78,  82,  66,  80,  81,  70,  76,  75,   1,
          69,  62],
        [ 65,   1,  84,  66,  75,  81,   1,  62,  84,  62,  86,   1,  81,  69,
          79,  76],
        [ 81,   1,  76,  75,  66,   1,  62,  75,  76,  81,  69,  66,  79,   1,
          82,  75],
        [ 70,  68,  70,  76,  82,  80,   1,  64,  62,  80,  81,  73,  66,  12,
           1,  84]])
when context is tensor([72]) target is 66
when context is tensor([72, 66]) target is 79
when context is tensor([72, 66, 79]) target is 116
when conte

### Simplest possible nn : bigram

In [7]:
# cross entropy / negative log likelihood
import math

def CEL(pred: list[float], true_idx: int):
    sum_exps = sum(math.exp(c) for c in pred)
    # compute softmax prob of the true class
    prob_true = math.exp(pred[true_idx])/sum_exps
    return -math.log(prob_true)

# tests
print(CEL([0.0, -100.0, -100.0], 0))
print(CEL([0.1, 2, 0.3], 1))
print(CEL([0.1, 0.2, 0.3], 2))

-0.0
0.28687085095710846
1.001942848229244


In [8]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    
    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lut
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, idx, targets=None):
        # idx and targets are both (batch_size (B), block_size (T)) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T, vocab_size (C)), this is the same as [idx], accessing the idxth row
        # pytorch expects a two dimensional object for the loss, i.e. instance * vocab_size
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx) # __call__ is defined as forward in nn.Module ?
            logits = logits[:, -1, :] # takes the last elt in Time (T) / context dim
            probs = F.softmax(logits, dim=-1) # converts to probs
            idx_next = torch.multinomial(probs, num_samples=1) # sample from the distribution
            idx = torch.cat((idx, idx_next), dim=1)
        
        return idx              

m = BigramLanguageModel(vocab_size=vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)
print(decode(m.generate(torch.zeros((1,1), dtype=torch.long), max_new_tokens=1000)[0].tolist()))

torch.Size([64, 120])
tensor(5.5098, grad_fn=<NllLossBackward0>)

èb-oö.ákóě”1L9qpFbo~œ-è,Rîí_
:ZI﻿à,xg”Kŏç-æà[gE’tèiôD/*4t,ěEK“
u03[W,îvP£à[{m:@”v/ně½=Aqm=EWRê]½FR-I8WNlPZcqJ?vëAá&%1SÔYH }/“]7áâiV{Ctd-“&'A;öî#(kh3n4sŏdA*'“!(CRKLözâ- ‘Y'y-B7b]é@m﻿ëçA,“—S ’K*WF;6ö‘V—>PA]óêc£@V<”g£'£*;_q8>S$hOg —LqO
öV.-:s-%Zuě<)7z fVnO
òb#çWŏÔLCBFWNl&āT-K#ON:!ëŏqiœvM<—/Ik!EWě{têg(gF/ í**'0"vO£{0OZ[$}V>y'SíçJWÔ]uâ½z!$o/eà[rGœ’WÔnîŏæëâhoyôUòé5ā8N‘~è8HöT *R’)ëâ{.U)>J>ó’{5u(pŏó6~~Sv9áâ(>(Mw£œczdZk&Z[/GSöxW‘ò.)Jsě‘ò£!8Pg)ôírAd”ö.}”é2(x@Ô[r!4s.iVqi$îáââL
S@fyb’nj’ 4.v‘ò{=pôKé=-A.ô" wVH0(Vî1dcm ch#Uî:NQ#ö&àkUòôSc[rá‘C!5Nvos8ŏ7?Z£6l!Gsòcz﻿Q$îTW;uěæ ògy“’£2z9“ 2ěæu~p*ěÔgEëgb;á.Z£F/îçáDlP96MwP“']Rét:!râ}"Z1F æàQ%têî5fn[fyrWQTy>jL?d$gB?/U—s4:KDóê.[0FlJX(Ptf.>ācOZBf#P[{)JPiNFn<œJgŏ&jLEL
 (i':b?F﻿qŏçRfnRSçtNx7êb-è34S”tZá.o3Ôê2L—R#ŏ&aX” a>Uy1N4wwêX]āF_&Ym1s>báMC=—òoâ-$y' 4#hZ3X2($hě2Slfŏô$Sf=kO'ěE(î}4SgbQ,ān]Iæí>;£—Y£ $(Ôò,8}'íEMc4%-t6/Cc}9~!I[èz3?DRW;}.vOöæI8“<ö½AHmgBIXoTr0Yŏ@mā;0M?;ôày'cW,sR_k U4dZ@à
{4[:n4çâLT]PPP

### Training the bigram, optimizer

In [9]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [10]:
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
print(loss.item())


2.417314291000366


In [11]:
print(decode(m.generate(torch.zeros((1,1), dtype=torch.long), max_new_tokens=1000)[0].tolist()))


NGuband  t eng ys theron d, thatl t einc. ht sad.'ěnoun. mred ahes, bl he
sondgr oçks Mre ho f ston tanc)}Siny

d nin ther
ce to veree mpre, s them wowalymong wesofithinouamitulpors irdvevanëqup-I ar"


th anoun $Anangomas. omy on arowe

idod orine han the We a ‘Wke.’ssiraphe thivat lind.
amo te,
wlfVEve conend g
y s HA den e, athomaOUwh y--ŏis tima s, towoveh I Heang; moy prstus t


cthe aye igbl t ad h thineis serlichiser n ochely  a<roustyou aly lerelllishepe so%6Zwhand ou conddinte
MBuis Nathan  blisy vanvod,
nt. outond l1$d brd?'1Thelorerraswhonche sitre “s bubeferat july)Lise Fathe we ounisound ilor theld ncerma bed NDey
ped ome f thayss,’
wishergh toe ac£the beted Weng inght senanthaw?’tild Jen, onglo abeve liof st an ollprte t isind htheasin Tera g Jof ofofed
‘Tind bshathor acoçVE” theerery ih d cencamearier te t s cofo h P=45.
I buns. m,
t s o d me apes,
t t ad frfutto—woucorishthe, opp:!’t cighefldge isk; ilwofun hugl? heextche tenimobemshe was s mslesmy ckevesiatountain I  

### The "trick" in self-attention ?

In [12]:
B, T, C = 4, 16, 128
# x = torch.randn(B, T, C)

# # bow = bag of words <=> averaging words
# xbow = torch.zeros((B, T, C))
# for b in range(B):
#     for t in range(T):
#         xprev = x[b,:t+1] # => shape is t, C
#         xbow[b,t] = torch.mean(xprev, 0)


# # can be done faster with matmul, take advantage of triangular matrices
# a = torch.tril(torch.ones(3, 3)) # builds a lower triangular matrix, 3x3 here
# a = a / torch.sum(a, 1, keepdim=True) # normalize rows
# b = torch.randint(0,10,(3,2)).float() # random 3x2 matrix
# c = a @ b

# can be done using softmax
# create the tril matrix to indicate positions to mask
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril==0, float('-inf'))
print(wei)
wei = torch.softmax(wei, dim=-1) #-1 is the last axis, so here we're softmaxing between columns = in the line direction
print(wei)

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0

In [15]:
# self-attention

torch.manual_seed(4242)
B, T, C = 4, 16, 32
x = torch.randn((B,T,C))

# one head
head_size = 16
query = nn.Linear(C, head_size, bias=False) # equivalent to the learned query matrix
key = nn.Linear(C, head_size, bias=False) # equivalent to the learned key matrix
value = nn.Linear(C, head_size, bias=False) # equivalent to the learned value matrix
q = query(x) # (B, T, 16)
k = key(x) # (B, T, 16)
wei = q @ k.transpose(-2, -1) * head_size ** -0.5 # (B, T, 16) @ (B, 16, T) -> (B, T, T)

tril = torch.tril(torch.ones(T,T))
wei = wei.masked_fill(tril==0, float('-inf'))
# print(wei[0])
wei = torch.softmax(wei, dim=-1) #-1 is the last axis, so here we're softmaxing between columns = in the line direction
# print(wei[0])

print(q.var())
print(k.var())
print(wei.var())

tensor(0.3256, grad_fn=<VarBackward0>)
tensor(0.3382, grad_fn=<VarBackward0>)
tensor(0.0103, grad_fn=<VarBackward0>)


tensor(0.9631)
tensor(1.0812)
tensor(1.0404)
